# Playfair Cipher

## Giới thiệu

Playfair Cipher là một hệ mật mã thay thế đa ký tự (digraph substitution cipher).
Thuật toán được phát minh bởi Charles Wheatstone vào năm 1854.

Khác với các hệ mật mã cổ điển như Caesar hay Vigenere,
Playfair mã hóa **từng cặp ký tự (digraph)** thay vì từng ký tự riêng lẻ.

Thuật toán sử dụng một **ma trận khóa 5×5** được tạo từ keyword.

Do bảng chữ cái có 26 chữ cái nên Playfair thường **gộp I và J** để tạo ma trận 25 ký tự.

## 2. Cơ sở lý thuyết

Playfair Cipher sử dụng **ma trận 5×5** được tạo từ keyword không sở hữu chữ cái lặp lại nào.

Ví dụ với keyword: **HUST**

<table style="border-collapse:collapse;text-align:center;font-size:14px">
<tr>
<td>H</td><td>U</td><td>S</td><td>T</td><td>A</td>
</tr>
<tr>
<td>B</td><td>C</td><td>D</td><td>E</td><td>F</td>
</tr>
<tr>
<td>G</td><td>I</td><td>K</td><td>L</td><td>M</td>
</tr>
<tr>
<td>N</td><td>O</td><td>P</td><td>Q</td><td>R</td>
</tr>
<tr>
<td>V</td><td>W</td><td>X</td><td>Y</td><td>Z</td>
</tr>
</table>

Plaintext sẽ được chia thành **các cặp ký tự**.

#### Quy tắc phân chia text:
1. Tách plaintext ra làm các cặp ký tự.
2. Nếu 1 cặp có **2 ký tự giống nhau:**
→ Chuyển 1 ký tự sang cặp sau, ký tự thứ 2 trong cặp hiện tại sẽ là **"X"**.
3. Nếu lẻ ra 1 ký tự cuối cùng:
→ Ghép ký tự đó với **"X"** thành 1 cặp hoàn chỉnh.

#### Ví dụ:
ABCCDE → AB CX CD EX

#### Quy tắc mã hóa:
1. Nếu hai chữ **cùng hàng**
→ lấy chữ **bên phải**
2. Nếu hai chữ **cùng cột**
→ lấy chữ **bên dưới**
3. Nếu hai chữ **khác hàng khác cột**
→ tạo hình chữ nhật và lấy hai góc còn lại

## 3. Ý tưởng thuật toán

### Bước 1: Tạo ma trận khóa

- nhập keyword
- thay J → I
- loại bỏ ký tự trùng
- thêm các chữ còn lại trong alphabet

### Bước 2: Chuẩn hóa plaintext

- chuyển về chữ hoa
- bỏ ký tự không phải chữ
- thay J → I

### Bước 3: Chia plaintext thành các cặp

Ví dụ:

BALLOON

→ BA LX LO ON

Nếu hai chữ giống nhau → chèn X

### Bước 4: Áp dụng quy tắc Playfair để mã hóa

### Bước 5: Ghép các cặp lại thành ciphertext

## 4. Cài đặt thuật toán

#### Tạo ma trận khóa

In [1]:
def create_matrix(keyword):
    alphabet = "ABCDEFGHIKLMNOPQRSTUVWXYZ"
    keyword = keyword.upper().replace("J", "I")

    matrix_input = keyword + "".join(a for a in alphabet if a not in keyword)

    matrix = [list(matrix_input[i : i + 5]) for i in range(0, 25, 5)]
    return matrix

#### Hàm mã hóa

In [2]:
def playfair_encrypt(plaintext, matrix):

    plaintext = plaintext.upper().replace("J", "I")
    plaintext = "".join(c for c in plaintext if c.isalpha())

    pairs = []
    i = 0

    while i < len(plaintext):
        a = plaintext[i]

        if i + 1 >= len(plaintext):
            pairs.append((a, "X"))
            i += 1

        elif plaintext[i + 1] == a:
            pairs.append((a, "X"))
            i += 1

        else:
            pairs.append((a, plaintext[i + 1]))
            i += 2

    print("Plaintext sau khi chia:", pairs)

    pos = {matrix[r][c]: (r, c) for r in range(5) for c in range(5)}

    encrypted = []

    for a, b in pairs:
        r1, c1 = pos[a]
        r2, c2 = pos[b]

        if r1 == r2:
            encrypted.append(matrix[r1][(c1 + 1) % 5] + matrix[r2][(c2 + 1) % 5])

        elif c1 == c2:
            encrypted.append(matrix[(r1 + 1) % 5][c1] + matrix[(r2 + 1) % 5][c2])

        else:
            encrypted.append(matrix[r1][c2] + matrix[r2][c1])

    return encrypted

#### Hàm giải mã

In [3]:
def playfair_decrypt(ciphertext, matrix):

    ciphertext = ciphertext.upper()
    ciphertext = "".join(c for c in ciphertext if c.isalpha())

    pairs = []
    for i in range(0, len(ciphertext), 2):
        pairs.append((ciphertext[i], ciphertext[i+1]))

    pos = {matrix[r][c]: (r, c) for r in range(5) for c in range(5)}

    decrypted = []

    for a, b in pairs:
        r1, c1 = pos[a]
        r2, c2 = pos[b]

        if r1 == r2:
            decrypted.append(matrix[r1][(c1 - 1) % 5] + matrix[r2][(c2 - 1) % 5])

        elif c1 == c2:
            decrypted.append(matrix[(r1 - 1) % 5][c1] + matrix[(r2 - 1) % 5][c2])

        else:
            decrypted.append(matrix[r1][c2] + matrix[r2][c1])

    return decrypted

In [4]:
while True:
    keyword = input("Nhap keyword:").upper().replace("J", "I")

    if len(set(keyword)) != len(keyword):
        print("Keyword khong hop le. Nhap lai.")
    else:
        break


matrix = create_matrix(keyword)

plaintext = input("Nhap plaintext:")

print("")
print("Ma tran:")
for row in matrix:
    print(row)
print("")

cipher = playfair_encrypt(plaintext, matrix)
print("→ Ciphertext:", " ".join(cipher))

ciphertext = "".join(cipher)

plain = playfair_decrypt(ciphertext, matrix)

print("")
print("Decrypt:", " ".join(plain))
print("→ Plaintext (sau khi giai ma):", " ".join(plain).replace(" ", ""))

Nhap keyword: HUST
Nhap plaintext: DO TRONG TUAN



Ma tran:
['H', 'U', 'S', 'T', 'A']
['B', 'C', 'D', 'E', 'F']
['G', 'I', 'K', 'L', 'M']
['N', 'O', 'P', 'Q', 'R']
['V', 'W', 'X', 'Y', 'Z']

Plaintext sau khi chia: [('D', 'O'), ('T', 'R'), ('O', 'N'), ('G', 'T'), ('U', 'A'), ('N', 'X')]
→ Ciphertext: CP AQ PO LH SH PV

Decrypt: DO TR ON GT UA NX
→ Plaintext (sau khi giai ma): DOTRONGTUANX


## 5. Minh họa quá trình mã hóa

Plaintext: **DO TRONG TUAN**  
Keyword: **HUST**

#### Ma trận khóa:

<table style="border-collapse:collapse;text-align:center;font-size:14px">
<tr>
<td>H</td><td>U</td><td>S</td><td>T</td><td>A</td>
</tr>
<tr>
<td>B</td><td>C</td><td>D</td><td>E</td><td>F</td>
</tr>
<tr>
<td>G</td><td>I</td><td>K</td><td>L</td><td>M</td>
</tr>
<tr>
<td>N</td><td>O</td><td>P</td><td>Q</td><td>R</td>
</tr>
<tr>
<td>V</td><td>W</td><td>X</td><td>Y</td><td>Z</td>
</tr>
</table>

#### Chia plaintext:
BACHKHOA → BA CH KH OA

#### Áp dụng thuật toán Playfair:
| Pair | Quy tắc | Cipher |
|-----|--------|--------|
| DO | hình chữ nhật | CP |
| TR | hình chữ nhật | AQ |
| ON | cùng cột | WO |
| GT | hình chữ nhật | LH |
| UA | cùng hàng | SH |
| NX | hình chữ nhật | PV |

#### Ciphertext:
CPAQWOLHSHPV

## 6. Minh họa quá trình giải mã

Ciphertext: **CPAQWOLHSHPV**
Keyword: **HUST**

#### Ma trận khóa:

<table style="border-collapse:collapse;text-align:center;font-size:14px">
<tr>
<td>H</td><td>U</td><td>S</td><td>T</td><td>A</td>
</tr>
<tr>
<td>B</td><td>C</td><td>D</td><td>E</td><td>F</td>
</tr>
<tr>
<td>G</td><td>I</td><td>K</td><td>L</td><td>M</td>
</tr>
<tr>
<td>N</td><td>O</td><td>P</td><td>Q</td><td>R</td>
</tr>
<tr>
<td>V</td><td>W</td><td>X</td><td>Y</td><td>Z</td>
</tr>
</table>

#### Chia ciphertext:
CPAQWOLHSHPV → CP AQ WO LH SH PV

#### Áp dụng thuật toán Playfair:
| Pair | Quy tắc | Plain |
|-----|------|------|
| CP | hình chữ nhật | DO |
| AQ | hình chữ nhật | TR |
| WO | cùng cột | ON |
| LH | hình chữ nhật | GT |
| SH | cùng hàng | UA |
| PV | hình chữ nhật | NX |

#### Plaintext:
DOTRONGTUANX